So from Xiaotian's data there are different conditions: expertise, code(visual stimulus) and rendering effect. And Xiaotian mentioned she take code as a random effect, so I am controlling following parametr: expertise and rendering effect.

We take following comparison:
- Expertise are fixed, comparing rendering effect
- Rendering effect, comparing expertise

|        |   r1   |   r2   |   r3   |
| ------ | ------ | ------ | ------ |
| Beginner  |  B_r1  |  B_r2  |  B_r3  |
| Intermediate |  I_r1  |  I_r2  |  I_r3  | 

Choose the algorithm to use

In [1]:
algo = "ScaSim"

In [2]:
from tools.path import setup_paths
from tools.auxiliary.parse_cr_data import parse_cr_data
import pandas as pd
import os

paths = setup_paths()
output_path = os.path.join(paths["output_path"], "processed_dataset", algo, "code_rendering")
parsed_data = parse_cr_data(False)

print(f"Finishing parsing CR data")
print(f"Number of expertise level: {len(parsed_data)} {list(parsed_data.keys())}")

# Get all expertise levels and rendering types
expertise_levels = list(parsed_data.keys())
rendering_types = list(parsed_data[expertise_levels[0]].keys())

print(f"Number of rendering types: {len(rendering_types)} {rendering_types}")

# Display the matrix structure
print("\nData matrix structure:")
for expertise in expertise_levels:
    for render in rendering_types:
        if render in parsed_data[expertise]:
            print(f"{expertise} - {render}: {len(parsed_data[expertise][render])} csv files")
        else:
            print(f"{expertise} - {render}: Not available")

Finishing parsing CR data
Number of expertise level: 2 ['Intermediate', 'Beginner']
Number of rendering types: 3 ['r1', 'r2', 'r3']

Data matrix structure:
Intermediate - r1: 22 csv files
Intermediate - r2: 22 csv files
Intermediate - r3: 22 csv files
Beginner - r1: 22 csv files
Beginner - r2: 22 csv files
Beginner - r3: 22 csv files


In [3]:
def filter_duplicates(df: pd.DataFrame):
    '''
    Filter dataframe to remove duplicate Shape values, keeping only the first occurrence.
    :param df: DataFrame from .csv file.

    :return: filtered DataFrame.
    '''
    if df.empty:
        return df
    
    df = df.copy()

    if algo != "MultiMatch":
        df = df.drop_duplicates(subset=['score'], keep='first')
        df = df[df['score'] != 0.0]
    else: 
        # Drop duplicates based only on Shape column 
        df = df.drop_duplicates(subset=['Shape'], keep='first')
        # Drop rows where Shape is 1.0
        df = df[df['Shape'] != 1.0]
    return df

Comparison: fix expertise, fix rendering

With this we can study how the average of participant with same condition looks like.

In [ ]:
import numpy as np
import multimatch_gaze as m
from tools.path_similarity.nld_cr import nld_cr
from tools.path_similarity.scasim_cr import scasim_cr

dimensions = ["Shape", "Direction", "Length", "Position", "Duration"]

for expertise in expertise_levels:
    for render in rendering_types:
        exps = parsed_data[expertise][render]
        print(f"{expertise} - {render}: {len(exps)} items")

        results = []

        for i in range(len(exps)):
            expertise_a, render_a, id_a, vec_a, code_a = exps[i]

            for j in range(len(exps)):
                if j < i:
                    continue
                expertise_b, render_b, id_b, vec_b, code_b = exps[j]

                if id_a == id_b:
                    continue

                if code_a != code_b:
                    continue

                row = {
                    'exp_a': id_a,
                    'exp_b': id_b,
                    'expertise_a': expertise_a,
                    'expertise_b': expertise_b,
                    'render_a': render_a,
                    'render_b': render_b,
                    'code_a': code_a,
                    'code_b': code_b
                }

                if algo == 'NLD':
                    nld_score = nld_cr(vec_a, vec_b, screensize=(1920, 1080), grid=(12, 8))
                    row["score"] = nld_score
                elif algo == 'ScaSim':
                    scasim_score = scasim_cr(vec_a, vec_b)
                    row["score"] = scasim_score
                else:
                    result = m.docomparison(vec_a, vec_b, screensize=[1920, 1080])
                    for dim, value in zip(dimensions, result):
                        row[dim] = value

                results.append(row)

        df = filter_duplicates(pd.DataFrame(results))

        output_dir = os.path.join(output_path,'fix_expertise_rendering')
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{expertise}_{render}_result.csv")
        df.to_csv(output_file, index=False)
        print(f"Saved {len(df)} comparisons to {output_file}")

Intermediate - r1: 22 items
Saved 72 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering\Intermediate_r1_result.csv
Intermediate - r2: 22 items
Saved 74 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering\Intermediate_r2_result.csv
Intermediate - r3: 22 items
Saved 70 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering\Intermediate_r3_result.csv
Beginner - r1: 22 items
Saved 75 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering\Beginner_r1_result.csv
Beginner - r2: 22 items
Saved 74 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering\Beginner_r2_result.csv
Beginner - r3: 22 items
Saved 70 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_exper

Comparison within expertise: Change expertise, fixed rendering

For this comparison we study on how expertise differs the similarity when having the same rendering condition.

In [ ]:
import multimatch_gaze as m

dimensions = ["Shape", "Direction", "Length", "Position", "Duration"]

for render in rendering_types:
    exps_list = []
    results = []
    for expertise in expertise_levels:
        exps = parsed_data[expertise][render]
        exps_list.append(exps)
    for expertise_idx_a in range(len(exps_list)):
        for expertise_idx_b in range(len(exps_list)):
            if expertise_idx_b < expertise_idx_a:
                continue
            print(f"  Comparing {expertise_levels[expertise_idx_a]} vs {expertise_levels[expertise_idx_b]} for rendering {render}")
            for i in range(len(exps_list[expertise_idx_a])):
                expertise_a, render_a, id_a, vec_a, code_a = exps_list[expertise_idx_a][i]
                for j in range(len(exps_list[expertise_idx_b])):
                    expertise_b, render_b, id_b, vec_b, code_b = exps_list[expertise_idx_b][j]

                    if code_a != code_b:
                        continue
                    
                    row = {
                        'exp_a': id_a,
                        'exp_b': id_b,
                        'expertise_a': expertise_a,
                        'expertise_b': expertise_b,
                        'render_a': render_a,
                        'render_b': render_b,
                        'code_a': code_a,
                        'code_b': code_b
                    }

                    if algo == 'NLD':
                        nld_score = nld_cr(vec_a, vec_b, screensize=(1920, 1080), grid=(12, 8))
                        row["score"] = nld_score
                    elif algo == 'ScaSim':
                        scasim_score = scasim_cr(vec_a, vec_b)
                        row["score"] = scasim_score
                    else:
                        result = m.docomparison(vec_a, vec_b, screensize=[1920, 1080])
                        for dim, value in zip(dimensions, result):
                            row[dim] = value

                    results.append(row)

    df = filter_duplicates(pd.DataFrame(results))
        
    output_dir = os.path.join(output_path, "fix_rendering")
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"{render}_result.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved {len(df)} comparisons to {output_file}")
    

  Comparing Intermediate vs Intermediate for rendering r1
  Comparing Intermediate vs Beginner for rendering r1
  Comparing Beginner vs Intermediate for rendering r1
  Comparing Beginner vs Beginner for rendering r1
Saved 303 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_rendering\r1_result.csv
  Comparing Intermediate vs Intermediate for rendering r2
  Comparing Intermediate vs Beginner for rendering r2
  Comparing Beginner vs Intermediate for rendering r2
  Comparing Beginner vs Beginner for rendering r2
Saved 302 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_rendering\r2_result.csv
  Comparing Intermediate vs Intermediate for rendering r3
  Comparing Intermediate vs Beginner for rendering r3
  Comparing Beginner vs Intermediate for rendering r3
  Comparing Beginner vs Beginner for rendering r3
Saved 301 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering

Comparison within expertise: Change rendering, fixed expertise

For this comparison we study on how rendering differs for participant with same expertise

In [ ]:
import multimatch_gaze as m

dimensions = ["Shape", "Direction", "Length", "Position", "Duration"]

for expertise in expertise_levels:
    exps_list = []
    results = []
    for render in rendering_types:
        exps = parsed_data[expertise][render]
        exps_list.append(exps)

    for render_idx_a in range(len(exps_list)):
        for render_idx_b in range(len(exps_list)):
            if render_idx_b < render_idx_a:
                continue
            print(f"  Comparing {rendering_types[render_idx_a]} vs {rendering_types[render_idx_b]} for expertise {expertise}")

            for i in range(len(exps_list[render_idx_a])):
                expertise_a, render_a, id_a, vec_a, code_a = exps_list[render_idx_a][i]
                for j in range(len(exps_list[render_idx_b])):
                    expertise_b, render_b, id_b, vec_b, code_b = exps_list[render_idx_b][j]

                    if code_a != code_b:
                        continue
                    
                    row = {
                        'exp_a': id_a,
                        'exp_b': id_b,
                        'expertise_a': expertise_a,
                        'expertise_b': expertise_b,
                        'render_a': render_a,
                        'render_b': render_b,
                        'code_a': code_a,
                        'code_b': code_b
                    }
                    
                    if algo == 'NLD':
                        nld_score = nld_cr(vec_a, vec_b, screensize=(1920, 1080), grid=(12, 8))
                        row["score"] = nld_score
                    elif algo == 'ScaSim':
                        scasim_score = scasim_cr(vec_a, vec_b)
                        row["score"] = scasim_score
                    else:
                        result = m.docomparison(vec_a, vec_b, screensize=[1920, 1080])
                        for dim, value in zip(dimensions, result):
                            row[dim] = value

                    results.append(row)

    df = filter_duplicates(pd.DataFrame(results))
        
    output_dir = os.path.join(output_path, "fix_expertise")
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"{expertise}_result.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved {len(df)} comparisons to {output_file}")
    

  Comparing r1 vs r1 for expertise Intermediate
  Comparing r1 vs r2 for expertise Intermediate
  Comparing r1 vs r3 for expertise Intermediate
  Comparing r2 vs r1 for expertise Intermediate
  Comparing r2 vs r2 for expertise Intermediate
  Comparing r2 vs r3 for expertise Intermediate
  Comparing r3 vs r1 for expertise Intermediate
  Comparing r3 vs r2 for expertise Intermediate
  Comparing r3 vs r3 for expertise Intermediate
Saved 693 comparisons to D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise\Intermediate_result.csv
  Comparing r1 vs r1 for expertise Beginner
  Comparing r1 vs r2 for expertise Beginner
  Comparing r1 vs r3 for expertise Beginner
  Comparing r2 vs r1 for expertise Beginner
  Comparing r2 vs r2 for expertise Beginner
  Comparing r2 vs r3 for expertise Beginner
  Comparing r3 vs r1 for expertise Beginner
  Comparing r3 vs r2 for expertise Beginner
  Comparing r3 vs r3 for expertise Beginner
Saved 693 comparisons to D:\Storage

In [7]:
from tools.auxiliary.merge_csv_files import merge_csv_files

dirs = [
    os.path.join(output_path, "fix_expertise"),
    os.path.join(output_path, "fix_expertise_rendering"),
    os.path.join(output_path, "fix_rendering")
]

for dir in dirs:
    merge_csv_files(dir)

Found 2 CSV files in D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise
  Read: Beginner_result.csv (693 rows)
  Read: Intermediate_result.csv (693 rows)
✓ Combined 1386 rows
✓ Saved to: D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise\combined_data.csv

Found 6 CSV files in D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering
  Read: Beginner_r1_result.csv (75 rows)
  Read: Beginner_r2_result.csv (74 rows)
  Read: Beginner_r3_result.csv (70 rows)
  Read: Intermediate_r1_result.csv (72 rows)
  Read: Intermediate_r2_result.csv (74 rows)
  Read: Intermediate_r3_result.csv (70 rows)
✓ Combined 435 rows
✓ Saved to: D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_expertise_rendering\combined_data.csv

Found 3 CSV files in D:\Storage\ETH\Thesis\Code\output\processed_dataset\ScaSim\code_rendering\fix_rendering
  Read: r1_result.csv (303 rows)
 